**Speech Recognition in Python using Google Speech API**

### Extended into an AI Voice Notes and Meeting Summarizer

This notebook is the research/demo companion of the Flask + React web app. It has six parts:

| Part | What it does |
|---|---|
| A | Original project: speech-to-text on an uploaded file with the Google Speech API (kept as it was) |
| B | Microphone recording inside Colab (fixed) |
| C | Reusable NLP modules: transcription, preprocessing, summarisation, keywords, action items, export |
| D | Optional: use the **Google Cloud** Speech-to-Text API with your own key |
| E | Run the full pipeline on audio and on a sample meeting |
| F | Download the notes as TXT and JSON |

**Run the cells from top to bottom.** Part C writes the same `voicenotes/` package that the website backend uses.

## Setup

In [ ]:
!pip -q install SpeechRecognition google-cloud-speech pydub yake scikit-learn transformers sentencepiece
# ffmpeg is pre-installed on Colab. Elsewhere install it with: sudo apt install ffmpeg

## Part A: Speech recognition with the Google Speech API (original project)

In [ ]:
import speech_recognition as sr
from google.colab import files

In [ ]:
recognizer = sr.Recognizer()

print("Recognizer created successfully!")

In [ ]:
uploaded = files.upload()

In [ ]:
filename = list(uploaded.keys())[0]

print("Uploaded file:", filename)

In [ ]:
with sr.AudioFile(filename) as source:
    audio = recognizer.record(source)

print("Audio file loaded successfully!")

In [ ]:
text = recognizer.recognize_google(audio)

print("Recognized Text:")
print(text)

## Part B: Record from the microphone (fixed)

**What was wrong in the original:** the JavaScript both *defined* and *immediately called* `recordAudio()`, so the microphone was requested twice, and the browser's recording (WebM/Opus) was saved with a `.wav` name, so ffmpeg could not read it. **Fix:** define the function once, save with the real `.webm` extension, convert with ffmpeg, and show a clear message if anything fails.

If you see *NotAllowedError: Permission denied*, click the lock/tune icon in the address bar, allow the microphone for `colab.research.google.com`, and re-run. Otherwise skip to Part C and use the uploaded file.

In [ ]:
from IPython.display import Javascript, display
from google.colab.output import eval_js
from base64 import b64decode
import subprocess

RECORD_JS = """
async function recordAudio(seconds) {
  const stream = await navigator.mediaDevices.getUserMedia({audio: true});
  const recorder = new MediaRecorder(stream);
  const chunks = [];
  recorder.ondataavailable = e => chunks.push(e.data);
  recorder.start();
  await new Promise(r => setTimeout(r, seconds * 1000));
  const stopped = new Promise(r => { recorder.onstop = r; });
  recorder.stop();
  await stopped;
  stream.getTracks().forEach(t => t.stop());
  const blob = new Blob(chunks, {type: recorder.mimeType || 'audio/webm'});
  return await new Promise(resolve => {
    const reader = new FileReader();
    reader.onloadend = () => resolve(reader.result);
    reader.readAsDataURL(blob);
  });
}
"""

def record_audio(seconds=10, out_wav="converted_audio.wav"):
    """Record from the browser microphone and return the path of a 16 kHz mono WAV file."""
    display(Javascript(RECORD_JS))
    print(f"Recording for {seconds} seconds... speak now.")
    data = eval_js(f"recordAudio({seconds})")
    with open("recorded_audio.webm", "wb") as f:
        f.write(b64decode(data.split(",")[1]))
    result = subprocess.run(
        ["ffmpeg", "-y", "-i", "recorded_audio.webm", "-acodec", "pcm_s16le",
         "-ar", "16000", "-ac", "1", out_wav],
        capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError("ffmpeg could not convert the recording:\n" + result.stderr[-400:])
    print("Recording saved:", out_wav)
    return out_wav

try:
    mic_file = record_audio(10)
    with sr.AudioFile(mic_file) as source:
        mic_audio = recognizer.record(source)
    print("Recognized Speech:", recognizer.recognize_google(mic_audio, language="en-IN"))
except sr.UnknownValueError:
    print("Could not understand the speech.")
except sr.RequestError as e:
    print("Google Speech API error:", e)
except Exception as e:
    print("Microphone step skipped:", e)
    print("(Allow microphone access for this site, or continue with the uploaded file.)")

## Part C: Reusable NLP modules

Each cell below writes one module of the `voicenotes` package. These files are identical to the ones in the website's `backend/voicenotes/` folder, so the notebook and the web app always behave the same.

In [ ]:
!mkdir -p voicenotes

In [ ]:
%%writefile voicenotes/audio_utils.py
"""Audio loading, normalisation and chunking helpers (pydub + ffmpeg).

Google's synchronous Speech-to-Text call accepts at most ~60 seconds of audio,
so long recordings are split at natural pauses into chunks of <= 50 seconds.
"""
import io
import shutil

from pydub import AudioSegment
from pydub.silence import split_on_silence

TARGET_RATE = 16000          # 16 kHz is what Google STT works best with
MAX_CHUNK_MS = 50_000        # stay safely below the 60 s sync limit
MIN_CHUNK_MS = 4_000         # merge tiny fragments into their neighbours


class AudioError(Exception):
    """Raised when an audio file cannot be read or is unusable."""


def ffmpeg_available() -> bool:
    return shutil.which("ffmpeg") is not None


def load_audio(path: str) -> AudioSegment:
    """Load any common format and convert to 16 kHz, mono, 16-bit PCM."""
    try:
        audio = AudioSegment.from_file(path)
    except FileNotFoundError:
        raise AudioError(
            "ffmpeg is not installed or not on PATH. Install it to read "
            "MP3/M4A/WebM files (WAV works without it)."
        )
    except Exception as exc:  # pydub raises several different error types
        raise AudioError(
            f"Could not read this audio file ({exc}). Try a WAV, MP3 or M4A file."
        )
    if len(audio) == 0 or audio.rms == 0:
        raise AudioError("The audio is empty or completely silent.")
    return audio.set_frame_rate(TARGET_RATE).set_channels(1).set_sample_width(2)


def split_audio(audio: AudioSegment, max_ms: int = MAX_CHUNK_MS):
    """Split at silences, cap chunk length, and merge very short pieces."""
    parts = split_on_silence(
        audio,
        min_silence_len=700,
        silence_thresh=audio.dBFS - 16,
        keep_silence=300,
    ) or [audio]

    pieces = []
    for part in parts:
        for i in range(0, len(part), max_ms):
            pieces.append(part[i:i + max_ms])

    chunks = []
    for piece in pieces:
        if chunks and len(chunks[-1]) < MIN_CHUNK_MS and len(chunks[-1]) + len(piece) <= max_ms:
            chunks[-1] = chunks[-1] + piece
        else:
            chunks.append(piece)
    return chunks


def to_wav_bytes(segment: AudioSegment) -> bytes:
    buf = io.BytesIO()
    segment.export(buf, format="wav")
    return buf.getvalue()

In [ ]:
%%writefile voicenotes/transcriber.py
"""Speech-to-text using Google.

Two engines share one interface:

* ``cloud`` - Google Cloud Speech-to-Text API (needs a service-account key,
  returns punctuation and capitalisation).
* ``free``  - the free Google Web Speech API through the SpeechRecognition
  library (this is what the original notebook used; no key needed, but the
  output has no punctuation).

``auto`` picks ``cloud`` when GOOGLE_APPLICATION_CREDENTIALS points to a
valid key file, otherwise ``free``.
"""
import io
import os

import speech_recognition as sr

from .audio_utils import load_audio, split_audio, to_wav_bytes


class TranscriptionError(Exception):
    """Raised for any user-facing transcription failure."""


def cloud_configured() -> bool:
    path = os.getenv("GOOGLE_APPLICATION_CREDENTIALS", "")
    return bool(path) and os.path.isfile(path)


def resolve_engine(engine: str = "auto") -> str:
    if engine in ("cloud", "free"):
        return engine
    return "cloud" if cloud_configured() else "free"


def _ensure_sentence(text: str) -> str:
    """Capitalise and terminate a chunk so downstream sentence splitting works."""
    text = text.strip()
    if not text:
        return ""
    text = text[0].upper() + text[1:]
    return text if text[-1] in ".!?" else text + "."


# --------------------------------------------------------------------------
# Google Cloud Speech-to-Text
# --------------------------------------------------------------------------
def _make_cloud_client():
    if not cloud_configured():
        raise TranscriptionError(
            "Google Cloud credentials not found. Set GOOGLE_APPLICATION_CREDENTIALS "
            "to your service-account JSON file, or use the free engine."
        )
    from google.cloud import speech
    return speech.SpeechClient()


def _cloud_chunk(client, wav_bytes: bytes, language: str) -> str:
    from google.api_core.exceptions import GoogleAPICallError
    from google.cloud import speech

    config = speech.RecognitionConfig(
        encoding=speech.RecognitionConfig.AudioEncoding.LINEAR16,
        sample_rate_hertz=16000,
        audio_channel_count=1,
        language_code=language,
        enable_automatic_punctuation=True,
    )
    try:
        response = client.recognize(
            config=config, audio=speech.RecognitionAudio(content=wav_bytes)
        )
    except GoogleAPICallError as exc:
        raise TranscriptionError(
            f"Google Cloud Speech-to-Text error: {exc.message}. Check that the "
            "Speech-to-Text API is enabled and billing is active for the project."
        )
    return " ".join(r.alternatives[0].transcript.strip() for r in response.results)


# --------------------------------------------------------------------------
# Free Google Web Speech API (original notebook approach)
# --------------------------------------------------------------------------
def _free_chunk(recognizer: sr.Recognizer, wav_bytes: bytes, language: str) -> str:
    with sr.AudioFile(io.BytesIO(wav_bytes)) as source:
        audio = recognizer.record(source)
    try:
        return recognizer.recognize_google(audio, language=language)
    except sr.UnknownValueError:
        return ""  # nothing intelligible in this chunk
    except sr.RequestError as exc:
        raise TranscriptionError(
            f"Could not reach the Google Speech service ({exc}). Check your internet connection."
        )


# --------------------------------------------------------------------------
def transcribe_file(path: str, language: str = "en-IN", engine: str = "auto") -> dict:
    """Transcribe an audio file of any length and return text + metadata."""
    engine = resolve_engine(engine)
    audio = load_audio(path)
    chunks = split_audio(audio)

    if engine == "cloud":
        client = _make_cloud_client()
        run = lambda wav: _cloud_chunk(client, wav, language)  # noqa: E731
    else:
        recognizer = sr.Recognizer()
        run = lambda wav: _free_chunk(recognizer, wav, language)  # noqa: E731

    texts = []
    for chunk in chunks:
        piece = run(to_wav_bytes(chunk))
        if piece:
            texts.append(_ensure_sentence(piece))

    if not texts:
        raise TranscriptionError(
            "No speech could be recognised. Use a clearer recording and check the language setting."
        )
    return {
        "text": " ".join(texts),
        "engine": engine,
        "language": language,
        "duration_sec": round(len(audio) / 1000, 1),
        "chunks": len(chunks),
    }

In [ ]:
%%writefile voicenotes/preprocessing.py
"""NLP text preprocessing.

Two views of the text are produced:

* ``clean_text``  - light cleaning that keeps natural sentences. Summarisation
  and action-item rules need real sentences, so stop-words are NOT removed here.
* ``tokens``      - lower-cased, punctuation-free, stop-word-free tokens used
  for statistics (and by the TF-IDF keyword fallback).
"""
import re

from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

FILLERS = re.compile(
    r"\b(?:u+m+|u+h+|e+r+m*|h+m+|a+h+|you know|i mean|kind of like)\b[,]?\s*",
    re.IGNORECASE,
)
REPEATED_WORD = re.compile(r"\b(\w+)(\s+\1\b)+", re.IGNORECASE)
SENTENCE_SPLIT = re.compile(r"(?<=[.!?])\s+")
TOKEN = re.compile(r"[a-z]+(?:'[a-z]+)?")


def split_sentences(text: str) -> list:
    return [s.strip() for s in SENTENCE_SPLIT.split(text) if s.strip()]


def clean_text(text: str) -> tuple:
    """Return (clean_text, number_of_filler_words_removed)."""
    text = re.sub(r"\s+", " ", text).strip()
    text, n_fill = FILLERS.subn("", text)
    text = REPEATED_WORD.sub(r"\1", text)                 # "the the" -> "the"
    text = re.sub(r"\s+([,.!?])", r"\1", text)             # no space before punctuation
    text = re.sub(r"\s+", " ", text).strip()

    sentences = split_sentences(text)
    fixed = []
    for s in sentences:
        s = s[0].upper() + s[1:]                           # capitalise
        if s[-1] not in ".!?":
            s += "."                                       # terminate
        fixed.append(s)
    return " ".join(fixed), n_fill


def tokenize(text: str, remove_stopwords: bool = True) -> list:
    tokens = TOKEN.findall(text.lower())
    if remove_stopwords:
        tokens = [t for t in tokens if t not in ENGLISH_STOP_WORDS and len(t) > 2]
    return tokens


def preprocess(text: str) -> dict:
    cleaned, n_fill = clean_text(text)
    sentences = split_sentences(cleaned)
    all_tokens = tokenize(cleaned, remove_stopwords=False)
    tokens = tokenize(cleaned)
    return {
        "clean_text": cleaned,
        "sentences": sentences,
        "tokens": tokens,
        "stats": {
            "words": len(all_tokens),
            "sentences": len(sentences),
            "fillers_removed": n_fill,
            "tokens_after_stopwords": len(tokens),
            "unique_terms": len(set(tokens)),
        },
    }

In [ ]:
%%writefile voicenotes/summarizer.py
"""Automatic summarisation.

Primary method: abstractive summarisation with a pretrained transformer
(default: DistilBART trained on CNN/DailyMail). Long transcripts are handled
hierarchically: split into chunks -> summarise each -> summarise the summaries.

Fallback: extractive summarisation (TF-IDF sentence scoring) which needs no
model download. Used automatically if the transformer cannot be loaded.
"""
import logging
import math
import os

from sklearn.feature_extraction.text import TfidfVectorizer

from .preprocessing import split_sentences

log = logging.getLogger(__name__)
DEFAULT_MODEL = "sshleifer/distilbart-cnn-12-6"
CHUNK_WORDS = 450          # ~600-700 tokens, safely inside a 1024-token window
_cache = {}


def _load_model():
    if "model" not in _cache:
        import torch  # noqa: F401  (fail early if torch is missing)
        from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

        name = os.getenv("SUMMARY_MODEL", DEFAULT_MODEL)
        log.info("Loading summarisation model %s ...", name)
        _cache["name"] = name
        _cache["tok"] = AutoTokenizer.from_pretrained(name)
        _cache["model"] = AutoModelForSeq2SeqLM.from_pretrained(name).eval()
    return _cache["tok"], _cache["model"], _cache["name"]


def preload():
    """Optional: call at start-up so the first request is not slow."""
    try:
        _load_model()
    except Exception as exc:  # noqa: BLE001
        log.warning("Could not preload summariser: %s", exc)


def _generate(text: str) -> str:
    import torch

    tok, model, name = _load_model()
    n_words = len(text.split())
    max_len = min(130, max(30, int(n_words * 0.5)))
    min_len = max(10, min(max_len - 5, int(n_words * 0.15)))
    prompt = "summarize: " + text if "t5" in name.lower() else text
    inputs = tok(prompt, return_tensors="pt", truncation=True,
                 max_length=min(getattr(tok, "model_max_length", 1024), 1024))
    with torch.no_grad():
        ids = model.generate(
            **inputs, num_beams=4, max_length=max_len, min_length=min_len,
            no_repeat_ngram_size=3, length_penalty=2.0, early_stopping=True,
        )
    return tok.decode(ids[0], skip_special_tokens=True).strip()


def _chunk_sentences(sentences, limit=CHUNK_WORDS):
    chunks, current, count = [], [], 0
    for s in sentences:
        w = len(s.split())
        if current and count + w > limit:
            chunks.append(" ".join(current))
            current, count = [], 0
        current.append(s)
        count += w
    if current:
        chunks.append(" ".join(current))
    return chunks


def abstractive_summary(text: str) -> str:
    chunks = _chunk_sentences(split_sentences(text))
    partial = [_generate(c) for c in chunks]
    combined = " ".join(partial)
    if len(chunks) > 1 and len(combined.split()) > 150:   # second pass
        return _generate(combined)
    return combined


def extractive_summary(text: str, max_sentences: int = 4) -> str:
    sentences = split_sentences(text)
    if len(sentences) <= 2:
        return text
    k = max(2, min(max_sentences, round(len(sentences) * 0.25)))
    try:
        tfidf = TfidfVectorizer(stop_words="english").fit_transform(sentences)
    except ValueError:                                     # only stop-words
        return " ".join(sentences[:k])
    # Sum of TF-IDF weights, damped by length so long sentences don't always win
    scores = [tfidf[i].sum() / math.sqrt(len(sentences[i].split()) or 1)
              for i in range(len(sentences))]
    top = sorted(sorted(range(len(sentences)), key=lambda i: -scores[i])[:k])
    return " ".join(sentences[i] for i in top)


def summarize(text: str) -> dict:
    """Return {summary, bullets, method, warning}."""
    words = len(text.split())
    mode = os.getenv("SUMMARIZER", "auto").lower()   # auto | abstractive | extractive
    warning = None

    if words < 30:
        summary, method = text, "passthrough (text is already short)"
    elif mode == "extractive":
        summary, method = extractive_summary(text), "extractive (TF-IDF sentence scoring)"
    else:
        try:
            summary = abstractive_summary(text)
            method = f"abstractive ({_cache.get('name', DEFAULT_MODEL)})"
        except Exception as exc:  # noqa: BLE001
            if mode == "abstractive":
                raise
            log.warning("Abstractive summariser unavailable: %s", exc)
            summary, method = extractive_summary(text), "extractive (TF-IDF sentence scoring)"
            warning = ("The pretrained summarisation model could not be loaded, so a "
                       "lighter extractive summary was used instead.")
    return {"summary": summary, "bullets": split_sentences(summary),
            "method": method, "warning": warning}

In [ ]:
%%writefile voicenotes/keywords.py
"""Keyword extraction with YAKE (primary) and TF-IDF (fallback)."""
import re

from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer

from .preprocessing import split_sentences


def _yake_keywords(text: str, top_n: int):
    import yake

    extractor = yake.KeywordExtractor(lan="en", n=2, dedupLim=0.7, top=top_n * 2)
    raw = extractor.extract_keywords(text)
    pairs = []
    for a, b in raw:                                   # tolerate (kw, score) or (score, kw)
        kw, score = (a, b) if isinstance(a, str) else (b, a)
        pairs.append((kw.lower().strip(), float(score)))
    return pairs                                       # YAKE: LOWER score = MORE relevant


def _tfidf_keywords(text: str, top_n: int):
    """Treat each sentence as a document so IDF is meaningful for one transcript."""
    docs = split_sentences(text) or [text]
    vec = TfidfVectorizer(stop_words="english", ngram_range=(1, 2))
    matrix = vec.fit_transform(docs)
    weights = matrix.sum(axis=0).A1
    terms = vec.get_feature_names_out()
    ranked = sorted(zip(terms, weights), key=lambda x: -x[1])[: top_n * 2]
    top = ranked[0][1] if ranked else 1
    return [(t, 1 - (w / top) * 0.8) for t, w in ranked]  # convert to "lower = better"


_TIME_WORDS = {"monday", "tuesday", "wednesday", "thursday", "friday", "saturday", "sunday",
               "today", "tomorrow", "tonight", "yesterday", "week", "weekly", "next", "asap"}


def _keep(kw: str) -> bool:
    words = re.findall(r"[a-z]+", kw)
    return (len(kw) > 2 and bool(words)
            and not all(w in ENGLISH_STOP_WORDS or w in _TIME_WORDS for w in words))


def extract_keywords(text: str, top_n: int = 10, method: str = "yake") -> list:
    """Return [{keyword, relevance (0-1, higher = better), method}]."""
    used, pairs = method, []
    if method == "yake":
        try:
            pairs = _yake_keywords(text, top_n)
        except Exception:  # noqa: BLE001  (yake missing or failed)
            pairs = []
        if not pairs:
            used = "tfidf"
    if used == "tfidf" or not pairs:
        used = "tfidf"
        try:
            pairs = _tfidf_keywords(text, top_n)
        except ValueError:
            return []

    # remove near-duplicates (a unigram contained in an already chosen phrase)
    chosen = []
    for kw, score in sorted(pairs, key=lambda p: p[1]):
        if _keep(kw) and not any(kw in c[0].split() or kw == c[0] for c in chosen):
            chosen.append((kw, score))
        if len(chosen) == top_n:
            break
    if not chosen:
        return []

    lo, hi = min(s for _, s in chosen), max(s for _, s in chosen)
    out = []
    for kw, s in chosen:
        rel = 1.0 if hi == lo else 1 - ((s - lo) / (hi - lo)) * 0.8
        out.append({"keyword": kw, "relevance": round(rel, 2), "method": used})
    return out

In [ ]:
%%writefile voicenotes/action_items.py
"""Rule-based meeting action-item identification.

Each sentence is scored on linguistic evidence of a commitment or request:

  +2  strong obligation cue   ("need to", "must", "make sure", "let's", "please", "follow up" ...)
  +1  weak cue                ("will", "going to", "should", "can you" ...)
  +1  a concrete action verb  (send, prepare, schedule, review, finish ...)
  +1  a responsible person    (named person, or I / we / you)
  +1  a deadline phrase       ("by Friday", "tomorrow", "end of the week" ...)

A sentence with score >= 3 is reported as an action item. Rules are used
instead of a trained classifier because no labelled meeting dataset is needed
and every decision can be explained.
"""
import re

STRONG = re.compile(
    r"\b(?:action item|to-?do|need(?:s|ed)? to|(?:has|have) to|must|make sure|"
    r"follow(?:ing)? up|remember to|let'?s|please|responsible for|assigned to|"
    r"due (?:on|by|date)|deadline)\b", re.I)
WEAK = re.compile(
    r"\b(?:will|going to|should|shall|can you|could you|would you|"
    r"(?:i|we)'ll|plan(?:s|ning)? to)\b", re.I)
VERBS = {
    "send", "prepare", "schedule", "review", "update", "finish", "complete", "submit",
    "share", "create", "call", "email", "mail", "fix", "check", "book", "arrange",
    "finalize", "finalise", "draft", "write", "test", "deploy", "design", "organize",
    "organise", "contact", "confirm", "plan", "research", "publish", "present", "set",
    "follow", "discuss", "prepare", "collect", "upload", "download", "install", "build",
    "implement", "analyze", "analyse", "approve", "assign", "buy", "order", "print",
    "ask", "remind", "meet", "setup", "complete", "document", "report", "decide",
    "investigate", "resolve", "prioritize", "coordinate", "clean", "revise", "verify",
}

_DAYS = r"(?:monday|tuesday|wednesday|thursday|friday|saturday|sunday)"
_MONTHS = (r"(?:jan(?:uary)?|feb(?:ruary)?|mar(?:ch)?|apr(?:il)?|may|june?|july?|aug(?:ust)?|"
           r"sep(?:t(?:ember)?)?|oct(?:ober)?|nov(?:ember)?|dec(?:ember)?)")
_PERIOD = r"(?:week|month|quarter|year|" + _DAYS + r")"
DEADLINE = re.compile(
    r"\b(?:by|before|until|till|on|due)\s+(?:the\s+)?(?:"
    r"(?:next|this|coming)\s+" + _PERIOD + r"|end of (?:the )?(?:day|week|month|quarter|year)|"
    r"eod|eow|today|tonight|tomorrow|" + _DAYS + r"|"
    r"\d{1,2}(?:st|nd|rd|th)(?:\s+(?:of\s+)?" + _MONTHS + r")?|"
    r"\d{1,2}\s+(?:of\s+)?" + _MONTHS + r"|" + _MONTHS + r"\s+\d{1,2}(?:st|nd|rd|th)?|"
    r"\d{1,2}(?::\d{2})?\s*(?:am|pm))\b"
    r"|\b(?:next|this|coming)\s+" + _PERIOD + r"\b"
    r"|\b(?:tomorrow|tonight|asap)\b"
    r"|\bend of (?:the )?(?:day|week|month)\b"
    r"|\bwithin\s+(?:\d+|an?|one|two|three|four|five|six|seven)\s+(?:hours?|days?|weeks?|months?)\b",
    re.I)

NAME_STOP = {
    "The", "This", "That", "These", "Those", "It", "He", "She", "They", "We", "You", "I",
    "There", "Here", "What", "Who", "When", "Where", "Why", "How", "And", "But", "So",
    "Then", "Also", "Okay", "Ok", "Well", "Now", "Today", "Tomorrow", "Meeting", "Project",
    "Report", "Work", "Rain", "Which", "Everything", "Nothing", "Please", "Maybe", "Next",
}
_NAME = r"([A-Z][a-z]{1,15})"
NAME_PATTERNS = [
    re.compile(r"\b" + _NAME + r"(?:\s+and\s+" + _NAME + r")?\s+(?:will|should|must|shall|needs? to|"
               r"has to|have to|is going to|are going to|can|could)\b"),
    re.compile(r"^" + _NAME + r",\s*(?:please|can you|could you|would you)\b"),
    re.compile(r"\b(?:assigned to|assign(?:ed)? it to|responsible:?|owner:?)\s+" + _NAME),
    re.compile(r"\b" + _NAME + r"\s+is responsible\b"),
]
URGENT = re.compile(r"\b(?:urgent(?:ly)?|asap|immediately|critical|high priority|top priority)\b", re.I)
LEADING = re.compile(r"^(?:okay|ok|so|alright|right|and|also|then|well|now)[,]?\s+", re.I)


def _assignee(sentence: str) -> str:
    for pat in NAME_PATTERNS:
        m = pat.search(sentence)
        if m:
            names = [g for g in m.groups() if g and g not in NAME_STOP]
            if names:
                return " & ".join(names)
    low = sentence.lower()
    if re.search(r"\b(?:can|could|would) you\b|\byou (?:should|need|must|will)\b", low):
        return "Person addressed"
    if re.search(r"\b(?:i|i'll|i'm)\b", low):
        return "Speaker (self)"
    if re.search(r"\b(?:we|we'll|let'?s|us|everyone)\b", low):
        return "Team"
    return "Unassigned"


def _score(sentence: str, assignee: str, deadline) -> int:
    score = 0
    if STRONG.search(sentence):
        score += 2
    if WEAK.search(sentence):
        score += 1
    words = set(re.findall(r"[a-z]+", sentence.lower()))
    if words & VERBS:
        score += 1
    if assignee != "Unassigned":
        score += 1
    if deadline:
        score += 1
    return score


def extract_action_items(sentences: list, threshold: int = 3) -> list:
    """Return [{task, assignee, deadline, priority, confidence}] sorted by confidence."""
    items, seen = [], set()
    for sent in sentences:
        sent = sent.strip()
        if len(sent.split()) < 3 or sent.endswith("?") and not re.search(r"can you|could you|would you", sent, re.I):
            continue
        m = DEADLINE.search(sent)
        deadline = m.group(0).strip() if m else None
        who = _assignee(sent)
        score = _score(sent, who, deadline)
        if score < threshold:
            continue
        task = LEADING.sub("", sent).strip()
        task = task[0].upper() + task[1:]
        key = task.lower()
        if key in seen:
            continue
        seen.add(key)
        items.append({
            "task": task,
            "assignee": who,
            "deadline": deadline,
            "priority": "High" if URGENT.search(sent) else "Normal",
            "confidence": round(min(score / 5, 1.0), 2),
        })
    return sorted(items, key=lambda i: -i["confidence"])

In [ ]:
%%writefile voicenotes/exporter.py
"""Build downloadable TXT and JSON notes."""
import json
import textwrap
from datetime import datetime


def build_notes(transcript: str, analysis: dict, meta: dict = None) -> dict:
    """Combine transcript + analysis + metadata into one structured document."""
    meta = meta or {}
    summary = analysis.get("summary", {})
    return {
        "title": meta.get("title") or "Voice Notes",
        "created_at": datetime.now().isoformat(timespec="seconds"),
        "source_file": meta.get("source_file"),
        "language": meta.get("language"),
        "speech_engine": meta.get("engine"),
        "transcript": transcript,
        "summary": {
            "text": summary.get("summary", ""),
            "bullets": summary.get("bullets", []),
            "method": summary.get("method", ""),
        },
        "keywords": analysis.get("keywords", []),
        "action_items": analysis.get("action_items", []),
        "stats": analysis.get("stats", {}),
    }


def to_json(notes: dict) -> str:
    return json.dumps(notes, indent=2, ensure_ascii=False)


def _wrap(text: str) -> str:
    return "\n".join(textwrap.fill(p, 88) for p in (text or "").split("\n"))


def to_txt(notes: dict) -> str:
    line = "=" * 64
    out = [line, notes["title"].upper(), line, f"Created : {notes['created_at']}"]
    if notes.get("source_file"):
        out.append(f"Source  : {notes['source_file']}")
    if notes.get("language"):
        out.append(f"Language: {notes['language']}")

    keywords = ", ".join(k["keyword"] for k in notes["keywords"]) or "(none)"
    out += ["", "SUMMARY", "-" * 7, _wrap(notes["summary"]["text"]) or "(none)",
            "", "KEYWORDS", "-" * 8, keywords,
            "", "ACTION ITEMS", "-" * 12]

    if notes["action_items"]:
        for i, item in enumerate(notes["action_items"], 1):
            deadline = item["deadline"] or "-"
            out.append(f"{i}. {item['task']}")
            out.append(f"   Owner: {item['assignee']} | Deadline: {deadline} | "
                       f"Priority: {item['priority']}")
    else:
        out.append("(none detected)")

    out += ["", "FULL TRANSCRIPT", "-" * 15, _wrap(notes["transcript"]), ""]
    return "\n".join(out)

In [ ]:
%%writefile voicenotes/pipeline.py
"""End-to-end orchestration: audio -> transcript -> NLP analysis."""
from . import action_items, keywords, preprocessing, summarizer
from .transcriber import transcribe_file


def analyze_text(text: str) -> dict:
    """Run preprocessing, summarisation, keyword and action-item extraction."""
    pre = preprocessing.preprocess(text)
    clean = pre["clean_text"]
    return {
        "clean_text": clean,
        "stats": pre["stats"],
        "summary": summarizer.summarize(clean),
        "keywords": keywords.extract_keywords(clean),
        "action_items": action_items.extract_action_items(pre["sentences"]),
    }


def transcribe_and_analyze(path: str, language: str = "en-IN", engine: str = "auto") -> dict:
    t = transcribe_file(path, language, engine)
    return {"transcription": t, "analysis": analyze_text(t["text"])}

In [ ]:
%%writefile voicenotes/__init__.py
"""voicenotes - reusable NLP modules for the AI Voice Notes & Meeting Summarizer.

The package has no Flask dependency, so the same code runs in the Flask API
and in the Google Colab notebook.
"""
from .pipeline import analyze_text, transcribe_and_analyze  # noqa: F401

## Part D (optional): Google Cloud Speech-to-Text API

Without a key the pipeline uses the free Google Web Speech API from Part A (**no punctuation** in the output). With a service-account key it uses the official **Google Cloud Speech-to-Text API**, which adds punctuation and capitalisation, so sentence splitting and action-item detection work noticeably better.

Create the key by following the *Google API setup* steps in the README, then run the cell and upload the JSON file. Skip it to stay on the free engine.

In [ ]:
import os
from google.colab import files

use_cloud = False   # <- set to True after you have your key.json
if use_cloud:
    key = files.upload()
    key_path = list(key.keys())[0]
    os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = os.path.abspath(key_path)
    print("Using Google Cloud Speech-to-Text with", key_path)
else:
    print("Using the free Google Web Speech API.")

## Part E: Run the full pipeline

In [ ]:
import importlib, json, os
import voicenotes
from voicenotes import analyze_text, transcribe_and_analyze
from voicenotes.transcriber import resolve_engine

# Set SUMMARIZER=extractive for an instant, download-free summary instead of the transformer
# os.environ["SUMMARIZER"] = "extractive"

print("Speech engine in use:", resolve_engine("auto"))
result = transcribe_and_analyze(filename, language="en-IN")     # 'filename' = the file uploaded in Part A
print("\nTRANSCRIPT:\n", result["transcription"]["text"])
print("\nSUMMARY:\n", result["analysis"]["summary"]["summary"])
print("\nKEYWORDS:", [k["keyword"] for k in result["analysis"]["keywords"]])
print("ACTION ITEMS:", result["analysis"]["action_items"] or "none (this clip is not a meeting)")

### Try it on a sample meeting

The uploaded clip is not a meeting, so here is a realistic transcript to show all three outputs.

In [ ]:
sample_meeting = (
    "Welcome everyone to the weekly project sync. The website redesign is almost done and the design team "
    "has finished the new landing page and the checkout flow. Priya will send the final mockups to the client by Friday. "
    "We need to fix the login bug before the release because it is blocking the testers. "
    "Rahul should prepare the demo slides for the review meeting next week. "
    "The marketing budget for the quarter was approved and the campaign will launch in November. "
    "Please schedule a follow up call with the vendor tomorrow. "
    "I'll update the project documentation by end of the day. "
    "The server migration went smoothly and performance improved by thirty percent. "
    "Let's finalize the pricing proposal on 25th October."
)
analysis = analyze_text(sample_meeting)

print("PREPROCESSING STATS:", analysis["stats"])
print("\nSUMMARY (" + analysis["summary"]["method"] + "):\n" + analysis["summary"]["summary"])
print("\nKEYWORDS:")
for k in analysis["keywords"]:
    print(f"  {k['keyword']:<28} {k['relevance']:.2f}")
print("\nACTION ITEMS:")
for a in analysis["action_items"]:
    print(f"  - {a['task']}\n      owner: {a['assignee']} | deadline: {a['deadline']} | priority: {a['priority']}")

## Part F: Download the notes (TXT and JSON)

In [ ]:
from google.colab import files
from voicenotes.exporter import build_notes, to_json, to_txt

notes = build_notes(sample_meeting, analysis, {"title": "Weekly Project Sync", "language": "en-IN"})
open("voice-notes.txt", "w", encoding="utf-8").write(to_txt(notes))
open("voice-notes.json", "w", encoding="utf-8").write(to_json(notes))

print(to_txt(notes)[:900], "...")
files.download("voice-notes.txt")
files.download("voice-notes.json")